<h1>Data Prep Notebook</h1>
<h3>Author: Mike Stanton</h3>
<h3>Date: December 1st, 2025</h3>
<h3>Overview: </h3>
This notebook is the first step in the Implicit ALS pipeline. 
The data is downloaded, cleaned, and split into training and test sets.
This notebook only needs to be run one time

<h1>1. Import Libraries</h1>

In [ ]:
# Install required packages if not available
import sys
!{sys.executable} -m pip install kagglehub pandas numpy matplotlib seaborn plotly

print("Packages installed successfully!")

# Data manipulation and analysis
import pandas as pd
import numpy as np

# Data loading utilities
import zipfile
import os
import kagglehub

print("Libraries imported successfully!")


Packages installed successfully!
Libraries imported successfully!
Checking for implicit library...
✅ implicit library is already available!


<h1>2. Load the Data</h1>
<p>Takes about 1 minute</p>

In [ ]:
# Download the dataset using kagglehub
dataset_path = kagglehub.dataset_download("andrewmvd/spotify-playlists")
print(f"Dataset downloaded to: {dataset_path}")

# Find the CSV file (it might be in a ZIP)
csv_files = []
zip_files = []

for file in os.listdir(dataset_path):
    file_path = os.path.join(dataset_path, file)
    if file.lower().endswith('.csv'):
        csv_files.append(file_path)
    elif file.lower().endswith('.zip'):
        zip_files.append(file_path)

print(f"Found {len(csv_files)} CSV files and {len(zip_files)} ZIP files")

# Load the data with robust error handling
df = None

if csv_files:
    # Direct CSV file found
    csv_file = csv_files[0]
    print(f"Loading CSV file: {csv_file}")
    try:
        df = pd.read_csv(
            csv_file,
            encoding='iso-8859-1',  # Permissive encoding
            sep=',',
            quotechar='"',
            escapechar='\\',
            engine='python',
            on_bad_lines='skip'
        )
    except Exception as e:
        print(f"Error loading CSV directly: {e}")

elif zip_files:
    # Extract and load from ZIP file
    zip_file = zip_files[0]
    print(f"Extracting from ZIP file: {zip_file}")
    try:
        with zipfile.ZipFile(zip_file, 'r') as z:
            csv_name = next((n for n in z.namelist() if n.lower().endswith('.csv')), None)
            if csv_name:
                print(f"Found CSV in ZIP: {csv_name}")
                with z.open(csv_name) as f:
                    df = pd.read_csv(
                        f,
                        encoding='iso-8859-1',
                        sep=',',
                        quotechar='"',
                        escapechar='\\',
                        engine='python',
                        on_bad_lines='skip'
                    )
            else:
                print("No CSV file found in ZIP archive")
    except Exception as e:
        print(f"Error extracting from ZIP: {e}")

if df is not None:
    print(f"Dataset loaded successfully!")
    print(f"Shape: {df.shape}")
    print(f"Columns: {list(df.columns)}")
else:
    print("Failed to load dataset")

# Clean column names - remove extra spaces and quotes
if df is not None:
    print("Cleaning column names...")
    print("Before:", list(df.columns))
    
    # Strip spaces and quotes from column names
    df.columns = [col.strip().strip('"') for col in df.columns]
    
    print("After:", list(df.columns))
    print("Column names cleaned successfully!")

# Basic dataset information
print("Dataset Info:")
print(f"Number of rows: {df.shape[0]:,}")
print(f"Number of columns: {df.shape[1]}")
print(f"Memory usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
print("\nColumn data types:")
print(df.dtypes)
print("\nMissing values:")
print(df.isnull().sum())

# Key statistics
stats = {
    'Total Records': df.shape[0],
    'Unique Users': df['user_id'].nunique() if 'user_id' in df.columns else 'N/A',
    'Unique Artists': df['artistname'].nunique() if 'artistname' in df.columns else 'N/A',
    'Unique Tracks': df['trackname'].nunique() if 'trackname' in df.columns else 'N/A',
    'Unique Playlists': df['playlistname'].nunique() if 'playlistname' in df.columns else 'N/A'
}

for key, value in stats.items():
    print(f"{key}: {value:,}" if isinstance(value, int) else f"{key}: {value}")


Dataset downloaded to: C:\Users\mstan\.cache\kagglehub\datasets\andrewmvd\spotify-playlists\versions\1
Found 1 CSV files and 0 ZIP files
Loading CSV file: C:\Users\mstan\.cache\kagglehub\datasets\andrewmvd\spotify-playlists\versions\1\spotify_dataset.csv
Dataset loaded successfully!
Shape: (12791243, 4)
Columns: ['user_id', ' "artistname"', ' "trackname"', ' "playlistname"']
Cleaning column names...
Before: ['user_id', ' "artistname"', ' "trackname"', ' "playlistname"']
After: ['user_id', 'artistname', 'trackname', 'playlistname']
Column names cleaned successfully!
Dataset Info:
Number of rows: 12,791,243
Number of columns: 4
Memory usage: 3390.16 MB

Column data types:
user_id         object
artistname      object
trackname       object
playlistname    object
dtype: object

Missing values:
user_id             0
artistname      33536
trackname          88
playlistname       41
dtype: int64
Total Records: 12,791,243
Unique Users: 15,910
Unique Artists: 287,438
Unique Tracks: 1,999,879
U

In [ ]:
df

,user_id,artistname,trackname,playlistname
0,9cc0cfd4d7d7885102480dd99e7a90d6,Elvis Costello,(The Angels Wanna Wear My) Red Shoes,HARD ROCK 2010
1,9cc0cfd4d7d7885102480dd99e7a90d6,Elvis Costello & The Attractions,"(What's So Funny 'Bout) Peace, Love And Unders...",HARD ROCK 2010
2,9cc0cfd4d7d7885102480dd99e7a90d6,Tiffany Page,7 Years Too Late,HARD ROCK 2010
3,9cc0cfd4d7d7885102480dd99e7a90d6,Elvis Costello & The Attractions,Accidents Will Happen,HARD ROCK 2010
4,9cc0cfd4d7d7885102480dd99e7a90d6,Elvis Costello,Alison,HARD ROCK 2010
...,...,...,...,...
12791238,2302bf9c64dc63d88a750215ed187f2c,MÃ¶tley CrÃ¼e,Wild Side,iPhone
12791239,2302bf9c64dc63d88a750215ed187f2c,John Lennon,Woman,iPhone
12791240,2302bf9c64dc63d88a750215ed187f2c,Tom Petty,You Don't Know How It Feels,iPhone
12791241,2302bf9c64dc63d88a750215ed187f2c,Tom Petty,You Wreck Me,iPhone


<h1>3. Clean The Artist and Tracknames for Consistency</h1>
takes > 10 minutes

In [ ]:
# Clean and standardize artistname and trackname 
import re

def clean_text(text):
    """
    Standardize text by:
    1. Converting to lowercase for consistency
    2. Standardizing common abbreviations 
    3. Removing extra punctuation and whitespace
    4. Handling special characters
    """
    if pd.isna(text) or text == '':
        return text
    
    # Convert to string and lowercase
    text = str(text).lower()
    
    # Fix common encoding corruptions first
    encoding_fixes = {
        'ã¡': 'á', 'ã©': 'é', 'ã­': 'í', 'ã³': 'ó', 'ãº': 'ú',
        'ã¤': 'ä', 'ã«': 'ë', 'ã¯': 'ï', 'ã¶': 'ö', 'ã¼': 'ü',
        'ã ': 'à', 'ã¨': 'è', 'ã¬': 'ì', 'ã²': 'ò', 'ã¹': 'ù',
        'ã¢': 'â', 'ãª': 'ê', 'ã®': 'î', 'ã´': 'ô', 'ã»': 'û',
        'ã§': 'ç', 'ã±': 'ñ', 'ã¿': 'ÿ'
    }
    
    for corrupted, correct in encoding_fixes.items():
        text = text.replace(corrupted, correct)
    
    # Remove extra whitespace and normalize
    text = re.sub(r'\s+', ' ', text.strip())
    
    # Common abbreviations standardization
    abbreviations = {
        r'\bft\.?\b': 'feat',
        r'\bfeat\.?\b': 'feat', 
        r'\bfeaturing\b': 'feat',
        r'\bw\/\b': 'with',
        r'\bw\b': 'with',
        r'\s*&\s*': ' and ',  # Match & with optional spaces around it
        r'\bvs\.?\b': 'vs',
        r'\bversus\b': 'vs',
        r'\bpt\.?\b': 'part',
        r'\bvol\.?\b': 'vol',
        r'\bno\.?\b': 'no',
        r'\bst\.?\b': 'st',
        r'\bdr\.?\b': 'dr',
        r'\bmr\.?\b': 'mr',
        r'\bms\.?\b': 'ms',
    }
    
    for pattern, replacement in abbreviations.items():
        text = re.sub(pattern, replacement, text)
    
    # Remove parentheses and everything inside them
    # This consolidates "Hey Jude (Remastered)" and "Hey Jude"
    text = re.sub(r'\([^)]*\)', '', text)
    
    # Remove or standardize punctuation
    # Keep important punctuation but remove excessive
    text = re.sub(r'[""''`\']', '', text)  # Remove various quote marks and apostrophes
    text = re.sub(r'[.]{2,}', '', text)  # Remove multiple dots
    text = re.sub(r'[-]{2,}', '-', text)  # Standardize multiple dashes
    text = re.sub(r'[!]{2,}', '!', text)  # Standardize multiple exclamation
    text = re.sub(r'[?]{2,}', '?', text)  # Standardize multiple questions
    text = re.sub(r'[,]{2,}', ',', text)  # Standardize multiple commas
    
    # Clean up spacing around punctuation
    text = re.sub(r'\s*,\s*', ', ', text)  # Standardize comma spacing
    text = re.sub(r'\s*-\s*', ' - ', text)  # Standardize dash spacing
    
    # Normalize accented characters to ASCII for better matching
    # This helps consolidate entries like "café" and "cafe"
    import unicodedata
    text = unicodedata.normalize('NFKD', text)
    text = ''.join(c for c in text if not unicodedata.combining(c))
    
    # Final cleanup
    text = re.sub(r'\s+', ' ', text.strip())
    
    return text

# Show before/after examples
print(" CLEANING ARTISTNAME AND TRACKNAME")
print("=" * 50)

# Sample some data to show what we're cleaning
print("Before cleaning - Sample artist names:")
sample_artists = df['artistname'].dropna().head(10).tolist()
for artist in sample_artists:
    print(f"  '{artist}'")

print("\nAfter cleaning - Same artists:")
for artist in sample_artists:
    cleaned = clean_text(artist)
    print(f"  '{artist}' → '{cleaned}'")

print(f"\nBefore cleaning - Sample track names:")
sample_tracks = df['trackname'].dropna().head(10).tolist()
for track in sample_tracks:
    print(f"  '{track}'")

print("\nAfter cleaning - Same tracks:")
for track in sample_tracks:
    cleaned = clean_text(track)
    print(f"  '{track}' → '{cleaned}'")

# Apply cleaning to the dataframe
print(f"\n Applying cleaning to full dataset...")
print(f"Original unique artists: {df['artistname'].nunique():,}")
print(f"Original unique tracks: {df['trackname'].nunique():,}")

# Create cleaned versions
df['artistname_clean'] = df['artistname'].apply(clean_text)
df['trackname_clean'] = df['trackname'].apply(clean_text)

print(f"Cleaned unique artists: {df['artistname_clean'].nunique():,}")
print(f"Cleaned unique tracks: {df['trackname_clean'].nunique():,}")

# Calculate reduction in uniqueness
artist_reduction = df['artistname'].nunique() - df['artistname_clean'].nunique()
track_reduction = df['trackname'].nunique() - df['trackname_clean'].nunique()

print(f"\n CLEANING IMPACT:")
print(f"Artists consolidated: {artist_reduction:,} ({artist_reduction/df['artistname'].nunique()*100:.1f}%)")
print(f"Tracks consolidated: {track_reduction:,} ({track_reduction/df['trackname'].nunique()*100:.1f}%)")

# Replace original columns with cleaned versions
df['artistname'] = df['artistname_clean']
df['trackname'] = df['trackname_clean']

# Drop the temporary cleaned columns
df = df.drop(['artistname_clean', 'trackname_clean'], axis=1)

# Create combined artist-track identifier
print(f"\n Creating combined artist-track identifiers...")
print(f"Combining: artistname + ' - ' + trackname")

# Create the combined identifier
df['trackname'] = df['artistname'] + ' - ' + df['trackname']

print(f"\nExample combined tracks:")
sample_combined = df['trackname'].dropna().head(5).tolist()
for track in sample_combined:
    print(f"  '{track}'")

# Show final statistics
print(f"\n FINAL STATISTICS:")
print(f"Unique combined artist-tracks: {df['trackname'].nunique():,}")
print(f"Unique users: {df['user_id'].nunique():,}")

print(f"\n Cleaning and combination complete!")
print(f"Final dataset shape: {df.shape}")
print(f"Now ready for recommendation system with user-track interactions!")


🧹 CLEANING ARTISTNAME AND TRACKNAME
Before cleaning - Sample artist names:
  'Elvis Costello'
  'Elvis Costello & The Attractions'
  'Tiffany Page'
  'Elvis Costello & The Attractions'
  'Elvis Costello'
  'Lissie'
  'Paul McCartney'
  'Joe Echo'
  'Paul McCartney'
  'Lissie'

After cleaning - Same artists:
  'Elvis Costello' → 'elvis costello'
  'Elvis Costello & The Attractions' → 'elvis costello and the attractions'
  'Tiffany Page' → 'tiffany page'
  'Elvis Costello & The Attractions' → 'elvis costello and the attractions'
  'Elvis Costello' → 'elvis costello'
  'Lissie' → 'lissie'
  'Paul McCartney' → 'paul mccartney'
  'Joe Echo' → 'joe echo'
  'Paul McCartney' → 'paul mccartney'
  'Lissie' → 'lissie'

Before cleaning - Sample track names:
  '(The Angels Wanna Wear My) Red Shoes'
  '(What's So Funny 'Bout) Peace, Love And Understanding'
  '7 Years Too Late'
  'Accidents Will Happen'
  'Alison'
  'All Be Okay'
  'Band On The Run'
  'Beautiful'
  'Blackbird - Live at CitiField, NYC

In [26]:
df.head(30)

,user_id,artistname,trackname,playlistname
0,9cc0cfd4d7d7885102480dd99e7a90d6,elvis costello,elvis costello - red shoes,HARD ROCK 2010
1,9cc0cfd4d7d7885102480dd99e7a90d6,elvis costello and the attractions,"elvis costello and the attractions - peace, lo...",HARD ROCK 2010
2,9cc0cfd4d7d7885102480dd99e7a90d6,tiffany page,tiffany page - 7 years too late,HARD ROCK 2010
3,9cc0cfd4d7d7885102480dd99e7a90d6,elvis costello and the attractions,elvis costello and the attractions - accidents...,HARD ROCK 2010
4,9cc0cfd4d7d7885102480dd99e7a90d6,elvis costello,elvis costello - alison,HARD ROCK 2010
5,9cc0cfd4d7d7885102480dd99e7a90d6,lissie,lissie - all be okay,HARD ROCK 2010
6,9cc0cfd4d7d7885102480dd99e7a90d6,paul mccartney,paul mccartney - band on the run,HARD ROCK 2010
7,9cc0cfd4d7d7885102480dd99e7a90d6,joe echo,joe echo - beautiful,HARD ROCK 2010
8,9cc0cfd4d7d7885102480dd99e7a90d6,paul mccartney,paul mccartney - blackbird - live at citifield...,HARD ROCK 2010
9,9cc0cfd4d7d7885102480dd99e7a90d6,lissie,lissie - bright side,HARD ROCK 2010


<h1> 4. Split the data for Training and Validation, save to CSV</h1>

In [ ]:
def final_stratified_split(df, output_dir='./data_splits/', test_size=0.2):
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
        
    train_path = os.path.join(output_dir, 'train_interactions.csv')
    val_path = os.path.join(output_dir, 'val_interactions.csv')

    # --- 1. Perform Stratified Split ---
    print(f"Splitting interactions for {df['user_id'].nunique():,} users...")
    
    # Randomize the whole dataframe once at the start
    df = df.sample(frac=1, random_state=42).reset_index(drop=True)
    
    # Group by user and split
    # We use a transform to create a mask for the first 80% of each user's rows
    user_counts = df.groupby('user_id').cumcount()
    user_totals = df.groupby('user_id')['user_id'].transform('count')
    
    # Mask for training: True if the interaction index is within the first 80%
    train_mask = user_counts < (user_totals * (1 - test_size)).astype(int)
    
    # Ensure users with 1 interaction stay in train
    train_mask = train_mask | (user_totals == 1)

    df_train = df[train_mask]
    df_val = df[~train_mask]

    # --- 2. The "Validation Purge" (Preventing Cold Start) ---
    print("Purging unknown items and users from validation set...")
    
    # Get sets of known entities from Training
    known_users = set(df_train['user_id'].unique())
    known_tracks = set(df_train['trackname'].unique())

    # Keep only validation records where BOTH user and track were seen in training
    initial_val_count = len(df_val)
    df_val = df_val[df_val['user_id'].isin(known_users) & 
                    df_val['trackname'].isin(known_tracks)]
    
    purged_count = initial_val_count - len(df_val)
    print(f"Removed {purged_count:,} validation records (unknown tracks/users).")

    # --- 3. Save to Disk ---
    print(f"Saving to {output_dir}...")
    df_train.to_csv(train_path, index=False)
    df_val.to_csv(val_path, index=False)
    
    print(f"\n FINAL SPLIT STATISTICS:")
    print(f"Train Interactions: {len(df_train):,}")
    print(f"Val Interactions:   {len(df_val):,}")
    print(f"Train Items:        {df_train['trackname'].nunique():,}")
    print(f"Val Items:          {df_val['trackname'].nunique():,}") # Should be <= Train Items

# Run the split
# final_stratified_split(df_cleaned)
final_stratified_split(df=df)


Splitting interactions for 15,910 users...
Purging unknown items and users from validation set...
Removed 316,437 validation records (unknown tracks/users).
Saving to ./data_splits/...

✅ FINAL SPLIT STATISTICS:
Train Interactions: 10,226,828
Val Interactions:   2,247,978
Train Items:        2,250,904
Val Items:          665,365
